## C-Drama RAG: Parser Prompt Lab

A scratchpad for iterating on `PARSE_SYSTEM_PROMPT` in `pipeline.py`.

| Prompt | Model | Purpose | Cost concern |
|---|---|---|---|
| `PARSE_SYSTEM_PROMPT` | `gpt-4o-mini` | Free-text → `QueryFilters` struct | Called on every message |

**What this notebook does:**
1. **Golden-set grading** — run a suite of queries, auto-score each field against expectations (not just eyeball the output)
2. **Search-mode classification** — stress the reference / semantic / sql boundary (the routing decision that matters most)
3. **Multi-turn history** — filter refinement, exclusion accumulation, pronoun resolution
4. **A/B prompt variants** — compare wording changes with a scored summary table
5. **Robustness** — typos, Chinese input, mixed languages, negation, slang
6. **Model comparison** — same cases across different models (4o-mini vs 4.1-mini vs 4.1-nano vs 4o)

For generator prompt work, see `04b_generator_lab.ipynb`.

---
## Setup

In [ ]:
import time

from openai import OpenAI

from src.recommender.pipeline import PARSE_SYSTEM_PROMPT, HISTORY_MESSAGES, MODEL
from src.recommender.models import QueryFilters
from src.env import load_secrets
from tests.fixtures.parser_cases import CASES, MODE_CASES

load_secrets()
client = OpenAI()

# Source: https://platform.openai.com/docs/pricing
MODEL_PRICING = {
    "gpt-4o-mini":  {"input": 0.15,  "output": 0.60},
    "gpt-4.1-mini": {"input": 0.40,  "output": 1.60},
    "gpt-4.1-nano": {"input": 0.10,  "output": 0.40},
    "gpt-4o":       {"input": 2.50,  "output": 10.00},
}


def token_cost(usage, model: str = MODEL) -> str:
    """Rough cost estimate in USD based on model pricing."""
    pricing = MODEL_PRICING.get(model, {"input": 0.15, "output": 0.60})
    input_cost  = (usage.prompt_tokens     / 1_000_000) * pricing["input"]
    output_cost = (usage.completion_tokens / 1_000_000) * pricing["output"]
    total = input_cost + output_cost
    return (
        f"in={usage.prompt_tokens} tok  "
        f"out={usage.completion_tokens} tok  "
        f"~${total:.6f}"
    )


def parse(
    query: str,
    history: list[dict] | None = None,
    prompt: str = PARSE_SYSTEM_PROMPT,
    model: str = MODEL,
):
    """Call the parser and return (QueryFilters, usage).

    Accepts model override so the same function works for both prompt
    A/B tests (Part 4) and model comparison (Part 6).
    """
    completion = client.chat.completions.parse(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            *(history or [])[-HISTORY_MESSAGES:],
            {"role": "user", "content": query},
        ],
        response_format=QueryFilters,
    )
    return completion.choices[0].message.parsed, completion.usage


def grade(result: QueryFilters, expect: dict) -> list[tuple[str, bool, str]]:
    """Grade a parser result against a dict of expectations.

    Each expectation value can be:
      - a plain value  -> exact equality check
      - a callable     -> must return truthy

    Returns a list of (field, passed, detail) tuples.
    """
    verdicts = []
    for field, expected in expect.items():
        actual = getattr(result, field)
        if callable(expected):
            ok = bool(expected(actual))
            detail = f"{actual!r}"
        else:
            ok = actual == expected
            detail = f"got {actual!r}, want {expected!r}"
        verdicts.append((field, ok, detail))
    return verdicts


def run_suite(
    cases: list[dict],
    prompt: str = PARSE_SYSTEM_PROMPT,
    model: str = MODEL,
    label: str = "",
    verbose: bool = True,
) -> dict:
    """Run a list of cases, print graded results, return summary stats.

    Each case is a dict with: id, query, expect, and optionally history.
    Set verbose=False to suppress per-case output (useful when running
    many model x prompt combinations in Part 6).
    """
    total = 0
    passed = 0
    failures = []
    total_cost = 0.0

    for case in cases:
        result, usage = parse(case["query"], case.get("history"), prompt, model)
        verdicts = grade(result, case["expect"])
        case_ok = all(ok for _, ok, _ in verdicts)

        pricing = MODEL_PRICING.get(model, {"input": 0.15, "output": 0.60})
        cost = (
            (usage.prompt_tokens / 1_000_000) * pricing["input"]
            + (usage.completion_tokens / 1_000_000) * pricing["output"]
        )
        total_cost += cost

        total += 1
        if case_ok:
            passed += 1
            status = "PASS"
        else:
            status = "FAIL"
            failures.append(case["id"])

        if verbose:
            prefix = f"[{label}] " if label else ""
            print(f"  {prefix}{status}  {case['id']}")
            print(f"         Q: {case['query']!r}")
            print(f"         -> {result.model_dump()}")
            for field, ok, detail in verdicts:
                mark = "+" if ok else "X"
                print(f"         [{mark}] {field}: {detail}")
            print(f"         {token_cost(usage, model)}")
            print()

    print(f"--- {label + ': ' if label else ''}{passed}/{total} passed  (total ~${total_cost:.4f}) ---")
    if failures:
        print(f"    Failed: {', '.join(failures)}")
    print()
    return {
        "label": label,
        "passed": passed,
        "total": total,
        "failures": failures,
        "total_cost": total_cost,
    }


print(f"Ready. Production model={MODEL}, HISTORY_MESSAGES={HISTORY_MESSAGES}.")
print(f"Models available for comparison: {list(MODEL_PRICING.keys())}")
print(f"Loaded {len(CASES)} golden + {len(MODE_CASES)} mode cases from tests/fixtures/parser_cases.py.")

---
## Part 1: Golden-set grading

Run a suite of queries against the **current production prompt** and auto-score
each field.  Every case has an `expect` dict — only the fields listed are checked
(partial assertions), and callable values (lambdas) allow flexible matching.

This replaces eyeballing raw output: you see `[+]` / `[X]` per field and a
pass/fail summary at the end.

In [ ]:
run_suite(CASES, label="golden")

---
## Part 2: Search-mode classification

The `search_mode` decision is the single most important parser output — it routes
the entire pipeline.  These cases stress the **boundaries** between modes:

- **reference vs semantic**: "like X" with plot elaboration — does the title win?
- **semantic vs sql**: plot description + structured filters — which signal wins?
- **description purity**: semantic-mode descriptions should capture plot cues only,
  not filter numbers like "2020" or "8.5" (the prompt says so explicitly).
- **cross-mode invariants**: `reference_title` must be None outside reference mode;
  `description` must be None outside semantic mode.

In [ ]:
run_suite(MODE_CASES, label="mode")

---
## Part 3: Multi-turn conversation

The pipeline passes conversation `history` to **both** the parser and the generator,
windowed to the last `HISTORY_MESSAGES` (currently 6) messages.

Tests here:
1. **Filter refinement across turns** — does the parser accumulate constraints from prior turns?
2. **Exclusion accumulation** — does "not that one" -> "not that one either" work?
3. **Pronoun resolution** — can the model resolve "I've already seen that one" back to the prior reference drama?
4. **History windowing** — with >6 messages, does old context correctly drop out?

In [ ]:
def parser_turn(query: str, history: list[dict], label: str = "") -> tuple[QueryFilters, list[dict]]:
    """Parse one turn, print result, return (result, updated_history)."""
    result, usage = parse(query, history)
    print(f"  [{label}] Q: {query!r}")
    print(f"         -> {result.model_dump()}")
    print(f"         {token_cost(usage)}\n")
    return result, history + [
        {"role": "user", "content": query},
        {"role": "assistant", "content": f"Here are picks based on: {result.model_dump()}"},
    ]


print("=== Session A: filter refinement ===")
h = []
r1, h = parser_turn("Something like Nirvana in Fire", h, "T1")
r2, h = parser_turn("Actually, make it from 2020 onwards", h, "T2")
r3, h = parser_turn("Also only highly rated please", h, "T3")
# Expected: T2 inherits reference_title from T1; T3 keeps min_year from T2 AND adds min_score
print(f"  Check T2 kept reference: {'nirvana' in (r2.reference_title or '').lower()}")
print(f"  Check T3 has min_year:   {r3.min_year}")
print(f"  Check T3 has min_score:  {r3.min_score}")
print()

print("=== Session B: exclusion accumulation ===")
h = []
_, h = parser_turn("Romance dramas, highly rated", h, "T1")
_, h = parser_turn("I already saw Story of Ming Lan -- not that one", h, "T2")
r3, h = parser_turn("And not Love Between Fairy and Devil either", h, "T3")
# Expected: T3 should carry both exclusions
print(f"  Check T3 excludes >= 2: {len(r3.exclude_titles)} titles -> {r3.exclude_titles}")
print()

print("=== Session C: pronoun resolution (hardest) ===")
h = []
_, h = parser_turn("Something similar to The Long Ballad", h, "T1")
r2, h = parser_turn("I've actually already seen that one -- something else please", h, "T2")
# Expected T2: exclude_titles should include "The Long Ballad" even though it's not named in T2
print(f"  Check T2 excludes 'long ballad': {any('long ballad' in x.lower() for x in r2.exclude_titles)}")
print()

print("=== Session D: history windowing ===")
# Build a history with 8 messages (> HISTORY_MESSAGES=6).
# First pair mentions The Untamed, which should fall outside the window.
h = [
    {"role": "user", "content": "Recommend something like The Untamed"},
    {"role": "assistant", "content": "Here are some wuxia dramas similar to The Untamed..."},
    {"role": "user", "content": "Actually I'd like something like Hidden Love"},
    {"role": "assistant", "content": "Hidden Love (2023) is a great youth romance..."},
    {"role": "user", "content": "More like that please"},
    {"role": "assistant", "content": "Here are similar youth romance dramas..."},
    {"role": "user", "content": "Any with higher ratings?"},
    {"role": "assistant", "content": "These are all rated above 8.5..."},
]
r, _ = parser_turn("something similar but more romance", h, "T5")
print(f"  Check ref is NOT 'untamed': {'untamed' not in (r.reference_title or '').lower()}")
print(f"  Check ref title exists:     {r.reference_title!r}")

---
## Part 4: A/B prompt variants

Small wording changes produce measurably different parser behaviour.  This section
runs the **same golden cases** against multiple prompt variants and compares
pass rates.

Workflow:
1. Define variants below (A = current production, B/C = candidates)
2. Run all variants against `GOLDEN_CASES` + `MODE_CASES`
3. Compare the scored summary — promote the winner to `pipeline.py`

In [ ]:
# Variant A — current production prompt (imported from pipeline.py)
PARSE_PROMPT_A = PARSE_SYSTEM_PROMPT

# Variant B — explicit null defaults + cleaner scoring language
PARSE_PROMPT_B = """\
You are a Chinese drama expert. Extract search parameters from the user's query.

First decide search_mode — one of:
- "reference": user names a specific drama as an anchor ("similar to X", "like X", "more like X"). Put X in reference_title.
- "semantic":  user describes plot / characters / vibe but cannot name a title. Put the description in the description field — capture plot/character cues only, NOT filters like year or rating.
- "sql":       user gives only structured filters — genre, year, rating — with no plot description and no reference title.

Field rules:
- reference_title: only set in reference mode. NEVER put a drama title in genres.
- description: only set in semantic mode. Leave null/empty if user did not describe a plot.
- genres: only explicit genre words (romance, historical, wuxia, fantasy, mystery, etc). Ignore vague words like 'good' or 'fun'.
- exclude_titles: any drama the user says they watched, finished, didn't enjoy, or wants skipped.
- min_year: 'after 20XX' = 20XX+1, 'from 20XX onwards' = 20XX, 'no older than 20XX' = 20XX.
- min_score: 'rating above X' = X, 'rating at least X' = X, 'highly rated'/'top rated' = 8.5, 'good rating' = 8.0.

Leave a field null/empty if the user did not specify it — do not guess.\
"""

# Variant C — few-shot examples
PARSE_PROMPT_C = """\
You are a Chinese drama expert. Extract search parameters from the user's query.

First decide search_mode — one of:
- "reference": user names a specific drama ("similar to X", "like X"). Put X in reference_title.
- "semantic":  user describes plot / characters / vibe without naming a title. Put the description in description — plot cues only, no filters.
- "sql":       user gives only structured filters (genre, year, rating), no plot, no title.

Examples:
User: "Something like Nirvana in Fire, highly rated"
-> search_mode="reference", reference_title="Nirvana in Fire", min_score=8.5

User: "A drama about a heroine with amnesia who is enemies with the hero"
-> search_mode="semantic", description="heroine with amnesia, enemies with the hero"

User: "Romance dramas after 2021, I already saw Joy of Life"
-> search_mode="sql", genres=["romance"], min_year=2022, exclude_titles=["Joy of Life"]

User: "Just surprise me with something good"
-> search_mode="sql" (all fields null/empty — 'good' is too vague)

Rules:
- reference_title: only set in reference mode. NEVER put a drama title in genres.
- description: only set in semantic mode. Plot/character cues only — no year/score numbers.
- genres: explicit genre words only.
- exclude_titles: dramas user watched, finished, or wants skipped.
- min_year: 'after 20XX' = 20XX+1, 'from 20XX onwards' = 20XX.
- min_score: 'rating above/at least X' = X, 'highly rated' = 8.5, 'good rating' = 8.0.\
"""

# --- Run all variants against the same cases ---

ALL_CASES = CASES + MODE_CASES

VARIANTS = {
    "A (current)":        PARSE_PROMPT_A,
    "B (explicit nulls)": PARSE_PROMPT_B,
    "C (few-shot)":       PARSE_PROMPT_C,
}

results = []
for name, prompt in VARIANTS.items():
    print(f"{'=' * 60}")
    print(f"  VARIANT: {name}")
    print(f"{'=' * 60}\n")
    stats = run_suite(ALL_CASES, prompt=prompt, label=name)
    results.append(stats)

# --- Summary comparison ---
print("\n" + "=" * 60)
print("  SUMMARY")
print("=" * 60)
for r in results:
    pct = (r["passed"] / r["total"] * 100) if r["total"] else 0
    bar = "#" * r["passed"] + "." * (r["total"] - r["passed"])
    print(f"  {r['label']:<22}  {r['passed']:>2}/{r['total']}  ({pct:5.1f}%)  [{bar}]")
    if r["failures"]:
        print(f"    failed: {', '.join(r['failures'])}")
print()
print("Promote the best variant back to PARSE_SYSTEM_PROMPT in pipeline.py.")

---
## Part 5: Language & typo robustness

Real users misspell drama titles, write in Chinese, or mix Chinese/English.

This matters because `reference_title` goes straight into a DB `ILIKE` search — if the
parser mangles it, the reference drama lookup fails silently. Test whether the parser
extracts a usable title even from imperfect input.

Also check the negation path: `QueryFilters.exclude_genres` was added so queries like
`"no romance"` or `"avoid wuxia"` stop being silently dropped. Verify the parser splits
INCLUDED vs EXCLUDED genres correctly — and that the same genre never appears in both.

In [ ]:
ROBUSTNESS_SUITE = [
    # (category, query, expected_reference_title or None)
    ("typo",      "Something like Nirvana in Fyre",                   "Nirvana in Fire"),
    ("typo",      "Recommend me Story of Minglan",                     "Story of Ming Lan"),
    ("typo",      "Love between fairy and divil",                      "Love Between Fairy and Devil"),
    ("chinese",   "推荐一部像《琅琊榜》这样的历史剧",                   "琅琊榜"),
    ("chinese",   "我想看高分古装剧，不要太老的",                        None),
    ("mixed",     "Something like 琅琊榜 with political intrigue",     "琅琊榜"),
    ("mixed",     "历史 romance, rating above 8, from 2020",           None),
    ("romanized", "Something like Lang Ya Bang",                       "Lang Ya Bang"),
    ("negation",  "Historical drama, no romance please",               None),   # should set exclude_genres=['romance']
    ("negation",  "Wuxia but not fantasy",                             None),   # should set exclude_genres=['fantasy']
    ("slang",     "that chef's kiss scheming politician energy drama", None),
    ("slang",     "the one where the genius guy does 5D chess politics",None),
]

print(f"{'Cat':<12} {'Query':<55} {'ref_title extracted':<30} {'genres':<20} {'exclude_genres':<18} score  year")
print("-" * 165)

for cat, query, expected_ref in ROBUSTNESS_SUITE:
    result, _ = parse(query)

    ref_ok = ""
    if expected_ref:
        # Loose check: does the extracted title contain a key word from the expected title?
        key_word = expected_ref.split()[0].lower()
        ref_ok = " +" if result.reference_title and key_word in result.reference_title.lower() else " X"

    print(
        f"{cat:<12} {query[:55]:<55} "
        f"{str(result.reference_title):<28}{ref_ok}  "
        f"{str(result.genres):<20} "
        f"{str(result.exclude_genres):<18} "
        f"{str(result.min_score):<7}"
        f"{result.min_year}"
    )

print()
print("Note: 'negation' rows should now populate exclude_genres (e.g. ['romance'])")
print("instead of silently dropping the excluded genre or leaking it into genres.")

---
## Part 6: Model comparison

The parser currently uses `gpt-4o-mini`.  Is there a better model for this task?

The parser is a single, small structured-output call on every user message — so
the tradeoffs are:
- **Accuracy** — does a bigger model fix the edge cases 4o-mini struggles with?
- **Latency** — the parser is on the critical path; every ms matters.
- **Cost** — called once per user message, so cost scales linearly with traffic.

Models to compare (skip GPT-5 — way overkill for structured extraction):

| Model | Input $/1M | Output $/1M | Notes |
|---|---|---|---|
| `gpt-4o-mini` | $0.15 | $0.60 | Current production. Cheap and fast. |
| `gpt-4.1-nano` | $0.10 | $0.40 | Even cheaper — but is it accurate enough? |
| `gpt-4.1-mini` | $0.40 | $1.60 | Newer mini. Better instruction following? |
| `gpt-4o` | $2.50 | $10.00 | 17x more expensive. Worth it if it nails edge cases. |

The question isn't "which model is smartest" — it's "which model gives the best
accuracy-per-dollar for this specific structured extraction task."

In [ ]:
# Models to compare.  Comment out any you want to skip to save cost/time.
COMPARE_MODELS = [
    "gpt-4o-mini",   # current production
    "gpt-4.1-nano",  # cheapest — can it handle structured extraction?
    "gpt-4.1-mini",  # newer mini — better instruction following?
    "gpt-4o",        # full-size — the accuracy ceiling
]

# Use the production prompt for all models.  If you found a winning prompt
# variant in Part 4, paste it here to test (prompt x model) combinations.
COMPARE_PROMPT = PARSE_SYSTEM_PROMPT

# Run shared golden cases + mode boundary cases.
# verbose=False keeps the output manageable — we only want the summary.
ALL_CASES = CASES + MODE_CASES

model_results = []
for model_name in COMPARE_MODELS:
    print(f"{'=' * 60}")
    print(f"  MODEL: {model_name}")
    print(f"{'=' * 60}")
    t0 = time.time()
    stats = run_suite(
        ALL_CASES,
        prompt=COMPARE_PROMPT,
        model=model_name,
        label=model_name,
        verbose=False,  # flip to True to see per-case detail for debugging
    )
    elapsed = time.time() - t0
    stats["elapsed"] = elapsed
    stats["cost_per_call"] = stats["total_cost"] / stats["total"] if stats["total"] else 0
    model_results.append(stats)

# --- Summary table ---
print("\n" + "=" * 70)
print("  MODEL COMPARISON SUMMARY")
print("=" * 70)
print(f"  {'Model':<18} {'Pass':>6} {'Rate':>7} {'Cost':>10} {'$/call':>10} {'Time':>8}")
print(f"  {'-'*18} {'-'*6} {'-'*7} {'-'*10} {'-'*10} {'-'*8}")

for r in model_results:
    pct = (r["passed"] / r["total"] * 100) if r["total"] else 0
    print(
        f"  {r['label']:<18} "
        f"{r['passed']:>2}/{r['total']:<3} "
        f"{pct:>5.1f}%  "
        f"${r['total_cost']:>8.4f}  "
        f"${r['cost_per_call']:>8.6f}  "
        f"{r['elapsed']:>5.1f}s"
    )
    if r["failures"]:
        print(f"    failed: {', '.join(r['failures'])}")

# --- Visual bar chart ---
print()
print("  Accuracy:")
for r in model_results:
    pct = (r["passed"] / r["total"] * 100) if r["total"] else 0
    bar = "#" * r["passed"] + "." * (r["total"] - r["passed"])
    print(f"  {r['label']:<18} [{bar}] {pct:.0f}%")

print()
print("  Decision guide:")
print("  - If nano matches mini's accuracy  -> switch to nano (save money)")
print("  - If 4.1-mini beats 4o-mini        -> switch to 4.1-mini (newer, same tier)")
print("  - If only 4o gets 100%             -> check which cases it fixes;")
print("    if they're edge cases, probably not worth the 17x cost increase")
print()
print("  To apply: update MODEL in src/recommender/pipeline.py,")
print("  then re-run: uv run pytest tests/parser/ -v")

---
## Notes & findings

### Part 1: Golden-set grading
- Overall: **54/57 golden + 9/9 mode = 63/66 (95.5%)** on the production prompt. Three persistent failures:
  - `score_good_rating` -- "Something with a good rating" gets refused instead of `min_score=8.0`. The prompt's score map fires on "highly rated" but not on the bare phrase "good rating".
  - `semantic_with_exclude` -- "Something about a contract marriage that turns real, I've already seen Well Intended Love" flips to **reference** mode (puts the title in `reference_title`) instead of staying **semantic** with the title only in `exclude_titles`.
  - `multiturn_followup_reference` -- bare follow-up "something older" after a prior reference turn gets refused instead of inheriting the reference from history.
- Multi-exclusion works (`exclude_multiple` passes -- 3 titles split correctly).
- "Just finished X" -> `exclude_titles` works (`exclude_just_finished` passes).
- All 7 injection / off-topic / harmful cases stay clean -- no leakage into genres or refs.

### Part 2: Search-mode classification
- **9/9 mode cases pass.** Boundaries hold up:
  - "like X + plot elaboration" stays in **reference** (the title wins).
  - "plot description + heavy filters" stays in **semantic** (filters don't drag it to sql).
  - Description fields stay free of year/score numbers.
  - Cross-mode invariants pass: no `reference_title` outside reference, no `description` outside semantic.

### Part 3: Multi-turn
- **Session A (filter refinement):** T2 inherited `reference_title=Nirvana in Fire`, T3 carried `min_year=2020` and `min_score=8.5`.
- **Session B (exclusion accumulation):** T3 carried both exclusions: `['Story of Ming Lan', 'Love Between Fairy and Devil']`.
- **Session C (pronoun resolution):** "I've actually already seen that one" correctly resolved back to "The Long Ballad" in `exclude_titles`. The hardest case -- works.
- **Session D (history windowing):** Out-of-window mention of "The Untamed" correctly dropped, parser picked the in-window "Hidden Love" as the reference.

### Part 4: A/B prompt wording
- **B (explicit nulls): 66/66 (100%)** -- clean sweep, **and cheaper** (~$0.0072 vs A's $0.0093, smaller prompt = fewer input tokens).
- A (current): 63/66 (95.5%) -- same 3 failures as Part 1.
- C (few-shot): 64/66 (97%) -- fixed nothing, broke `semantic_description_has_no_filters` and `mode_sem_with_filters`. The "Romance dramas after 2021" few-shot example primes the model to flip semantic->sql whenever year/score filters appear, even when there's a plot description.
- "good rating" -> only B sets `min_score=8.0`. A and C refuse.
- "highly rated" / "masterpiece wuxia" -> all variants set `min_score=8.5` correctly.
- **Action: promote variant B to `PARSE_SYSTEM_PROMPT`.** Few-shot was a regression -- the explicit-null phrasing is the actual win.

### Part 5: Language & typo robustness
- Typos preserved as-is in `reference_title` ("Nirvana in Fyre", "Story of Minglan", "Love Between Fairy and Devil"). Postgres `ILIKE` will still match the fixed cases; "Fyre" would miss in fuzzy search but gracefully degrades to the candidate fallback.
- Chinese pure: 《琅琊榜》→ `reference_title='琅琊榜'` (book brackets cleanly stripped). The query "我想看高分古装剧，不要太老的" extracts `genres=['historical']`, `min_score=8.0`, `min_year=2020` — semantic understanding of "高分" and "不要太老" works.
- Mixed CN/EN: extracts the Chinese title from an English-context query; `历史 romance, rating above 8, from 2020` gets all three filters (genres + min_score + min_year).
- Romanized pinyin "Lang Ya Bang" -> preserved as ref_title.
- **Negation populates `exclude_genres` correctly in both cases** with no leakage into `genres` -- the `exclude_genres` field is doing its job.

### Part 6: Model comparison

| Model | Pass | Cost | $/call | Time |
|---|---|---|---|---|
| gpt-4o-mini (current) | 63/66 (95.5%) | $0.0093 | $0.000142 | 123.5s |
| gpt-4.1-nano | 54/66 (81.8%) | $0.0063 | $0.000096 | 71.6s |
| gpt-4.1-mini | 63/66 (95.5%) | $0.0250 | $0.000379 | 79.3s |
| gpt-4o | 66/66 (100%) | $0.1556 | $0.002358 | 60.6s |

- **gpt-4.1-nano** is cheapest but loses 14 percentage points -- fails mode boundaries, `exclude_genre_*`, and multi-turn cases. Not viable.
- **gpt-4.1-mini** matches 4o-mini's accuracy but fails on a *different* set of cases (`sql_genre_only`, `sql_wuxia`, `mode_genre_only_is_sql`). Same accuracy at 2.7x cost -- no win.
- **gpt-4o** is the only model that hits 100% on the production prompt -- but variant B + 4o-mini also hits 100% at ~1/20th the cost.
- **Action: keep `MODEL=gpt-4o-mini` and switch the prompt to variant B.** That's a 100%-pass setup at the lowest tested price.
- Latency note: 4.1-mini and 4o are noticeably faster than 4o-mini at the suite level (~70-80s vs 123s for 66 calls) but per-call latency is sub-2s for all of them -- not a UX issue.

### Decisions made
- Promote prompt **variant B (explicit nulls)** to `PARSE_SYSTEM_PROMPT` in `pipeline.py`.
- Keep `MODEL = "gpt-4o-mini"`.
- After applying: re-run `uv run pytest tests/parser/ -v` to confirm the regression gate still passes.